In [ ]:
# --- LPA repo path bootstrap (added during repository reorganization) ---
# Makes this notebook find the project source (src/) and the data folders at the
# repository root, no matter which notebooks/<topic>/ subfolder it lives in.
# Run this cell first.
import os, sys

_root = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(_root, "pyproject.toml")) and os.path.dirname(_root) != _root:
    _root = os.path.dirname(_root)
os.chdir(_root)                       # so relative data paths (e.g. "Dataset/...") resolve
_src = os.path.join(_root, "src")
if _src not in sys.path:
    sys.path.insert(0, _src)          # so `from LPA import ...` works without installing
print("working dir set to repo root:", _root)


In [1]:
import pandas as pd
from LPA import Corpus, sockpuppet_distance , PCA
import altair as alt
alt.data_transformers.disable_max_rows()
import os
from typing import List
from visualize import plot_pca
import re
from collections import Counter
import pandas as pd
import csv
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns
import zipfile
from bs4 import BeautifulSoup
import shutil
import html2text
import re
import ast
import scipy.stats as stats
from sklearn.metrics import f1_score, precision_score, recall_score
import random
import datetime


Presidential EDA

In [4]:
path = f"Dataset\Pres\RAW"

In [13]:
pres_df = pd.read_csv(path+"/presidential_speeches.csv")

In [6]:
pres_df

,Date,President,Party,Speech Title,Summary,Transcript,URL
0,1789-04-30,George Washington,Unaffiliated,First Inaugural Address,Washington calls on Congress to avoid local an...,Fellow Citizens of the Senate and the House of...,https://millercenter.org/the-presidency/presid...
1,1789-10-03,George Washington,Unaffiliated,Thanksgiving Proclamation,"At the request of Congress, Washington establi...",Whereas it is the duty of all Nations to ackno...,https://millercenter.org/the-presidency/presid...
2,1790-01-08,George Washington,Unaffiliated,First Annual Message to Congress,"In a wide ranging speech, President Washington...",Fellow Citizens of the Senate and House of Rep...,https://millercenter.org/the-presidency/presid...
3,1790-12-08,George Washington,Unaffiliated,Second Annual Message to Congress,Washington focuses on commerce in his second a...,Fellow citizens of the Senate and House of Rep...,https://millercenter.org/the-presidency/presid...
4,1790-12-29,George Washington,Unaffiliated,Talk to the Chiefs and Counselors of the Senec...,The President reassures the Seneca Nation that...,"I the President of the United States, by my ow...",https://millercenter.org/the-presidency/presid...
...,...,...,...,...,...,...,...
987,2019-01-19,Donald Trump,Republican,Remarks about the US Southern Border,President Donald Trump speaks about what he se...,"Just a short time ago, I had the honor of pres...",https://millercenter.org/the-presidency/presid...
988,2019-02-05,Donald Trump,Republican,State of the Union Address,"In his second State of the Union Address, Pres...","Madam Speaker, Mr. Vice President, Members of ...",https://millercenter.org/the-presidency/presid...
989,2019-02-15,Donald Trump,Republican,Speech Declaring a National Emergency,President Donald Trump declares a national eme...,"Thank you very much, everybody. Before we begi...",https://millercenter.org/the-presidency/presid...
990,2019-09-24,Donald Trump,Republican,Remarks at the United Nations General Assembly,President Donald Trump speaks to the 74th sess...,"Thank you very much. Mr. President, Mr. Secret...",https://millercenter.org/the-presidency/presid...


In [17]:
def export_df_to_csvs(df, output_folder, author="President" , topic="Speech Title"):
    """
    Exports a DataFrame to individual CSV files, one for each row.

    Args:
    df: The DataFrame to be exported.
    output_folder: The path to the output folder.
    author: The name of the column that contains the author data.
    topic: The name of the column that contains the topic data.

    """

    for index, row in df.iterrows():
        author_value = row[author]
        topic_value = row[topic]

        filename = f'{author_value} - {topic_value}.csv'
        if len(filename) > 80:
            filename = filename[:80]+'.csv'
        filepath = os.path.join(output_folder, filename)

#         df.iloc[index].to_csv(filepath, index=False, columns=df.columns)
        print(type(df.iloc[index]))


In [18]:
export_df_to_csvs(pres_df , path+"/speeches_separated")

<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.S

<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.S

In [34]:
import os
import pandas as pd
import re

def export_rows_as_csv(dataframe, output_folder, author='author', topic='topic'):
    # Create the output folder if it doesn't exist
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # Iterate over each row in the dataframe
    for index, row in dataframe.iterrows():
        # Extract the author and topic values from the row
        row_author = row.get(author, 'author')
        row_topic = row.get(topic, 'topic')

        # Replace forward slashes in the author and topic with underscores
        row_author = re.sub(r'[/\\]', '_', row_author)
        row_topic = re.sub(r'[/\\]', '_', row_topic)
        # Create a new dataframe with a single row
        new_dataframe = pd.DataFrame([row], columns=dataframe.columns)

        # Generate the output filename
        filename = f"{row_author} - {row_topic}.csv"
        if len(filename) > 50:
            filename = filename[:50]+'.csv'
        filename = re.sub(r'[<>:"/\\|?*]', '_', filename)
        output_path = os.path.join(output_folder, filename)
        if output_path.endswith('.csv'):
            # Export the new dataframe to CSV
            new_dataframe.to_csv(output_path, index=False)
        else:
            new_dataframe.to_csv(output_path+".csv", index=False)


In [35]:
export_rows_as_csv(pres_df , path+"/speeches_separated",author="President" , topic="Speech Title")